# Réplica paso 2: LLM abierto local (prompting y active learning)

Como no hay API key de OpenAI, se reemplaza GPT-3.5 por **Qwen2.5-Coder-3B-Instruct**, un LLM abierto que corre dentro de Colab. Es una **desviación declarada** respecto al paper.

Activa la GPU T4. Si ya tienes abierta la sesión del paso 1, la carpeta del repo ya existe; si es una sesión nueva, ejecuta primero la celda de clonado.

In [ ]:
!nvidia-smi

## 0. Traer los cambios nuevos

Si la sesión del paso 1 sigue abierta, usa esta celda (conserva tus resultados):

In [ ]:
%cd /content/replica-data-leakage-detection
!git pull

Si es una **sesión nueva**, usa esta otra (clona y entrenaría de cero los resultados del paso 1):

In [ ]:
# SOLO si es sesión nueva:
# !git clone https://github.com/priscilaGO/replica-data-leakage-detection.git
# %cd replica-data-leakage-detection
# !pip install -q "transformers>=4.40,<5" tenacity scikit-learn pandas nltk

In [ ]:
!pip install -q accelerate

## A. Prompting con LLM local (0, 1, 3 y 5 ejemplos)

Primero una prueba rápida con 25 fragmentos para comprobar que funciona (1 a 2 min):

In [ ]:
%cd GPT4Code
!python prompt_local.py --shots 0 --limit 25

Si salió un diccionario con precision, recall y f2 sin errores, corre las cuatro configuraciones completas (381 fragmentos cada una):

In [ ]:
for k in [0, 1, 3, 5]:
    !python prompt_local.py --shots {k}

In [ ]:
import pandas as pd, glob
rows = [pd.read_csv(f).iloc[0] for f in sorted(glob.glob('../Results/GPT/local_*shot_metrics.csv'))]
display(pd.DataFrame(rows)[['shots','precision','recall','f1','f2']].round(2))
print('Paper (GPT-3.5): 0-shot F2 0.18 | 1-shot 0.30 | 3-shot 0.39 | 5-shot 0.31')

## B. Active learning con aumento de datos local

Tarda mucho (1 a 2 horas). Ejecutar solo cuando A haya terminado. Usa N = 50 y paciencia 5, como el paper.

In [ ]:
%cd ../AL4Code
!AUG_BACKEND=local python demo.py --active_learning --with_augmentation --n_init_labeled 50 --n_query 50 --patient 5 --seed 1 --tag al_seed1

In [ ]:
d = pd.read_csv('../Results/LeastConfidence_al_seed1.csv')
display(d[['round','precision','recall','f2','training_size']])
print('Ultima ronda:'); print(d.iloc[-1][['precision','recall','f2']])
print('Mejor F2 en alguna ronda:', d.f2.max())
print('Paper (Tabla 3, active learning): precision 0.86, recall 0.70, F2 0.72, 698 etiquetas')